In [4]:
import pandas as pd

df = pd.read_json("/jeopardy.json")
df.head()

,category,air_date,question,value,answer,round,show_number
0,HISTORY,2004-12-31,"'For the last 8 years of his life, Galileo was...",$200,Copernicus,Jeopardy!,4680
1,ESPN's TOP 10 ALL-TIME ATHLETES,2004-12-31,'No. 2: 1912 Olympian; football star at Carlis...,$200,Jim Thorpe,Jeopardy!,4680
2,EVERYBODY TALKS ABOUT IT...,2004-12-31,'The city of Yuma in this state has a record a...,$200,Arizona,Jeopardy!,4680
3,THE COMPANY LINE,2004-12-31,"'In 1963, live on ""The Art Linkletter Show"", t...",$200,McDonald\'s,Jeopardy!,4680
4,EPITAPHS & TRIBUTES,2004-12-31,"'Signer of the Dec. of Indep., framer of the C...",$200,John Adams,Jeopardy!,4680


In [5]:
df.columns

Index(['category', 'air_date', 'question', 'value', 'answer', 'round',
       'show_number'],
      dtype='object')

In [6]:
df.shape

(216930, 7)

In [7]:
df['value'].value_counts()

,count
value,
$400,42244
$800,31860
$200,30455
$600,20377
$1000,19539
...,...
"$4,238",1
"$16,400",1
"$1,347",1


In [9]:
df['value'] = df['value'].replace(r'[\$,]', '', regex=True)
df['value'] = pd.to_numeric(df['value'], errors='coerce')

df['value'].head()

,value
0,200.0
1,200.0
2,200.0
3,200.0
4,200.0


In [10]:
df.isnull().sum()

,0
category,0
air_date,0
question,0
value,3634
answer,0
round,0
show_number,0


In [11]:
df = df.dropna(subset=['value'])

df.shape

(213296, 7)

In [12]:
df['difficulty'] = df['value'].apply(
    lambda x: 'low' if x <= 600 else 'high'
)

df['difficulty'].value_counts()

,count
difficulty,
low,119805
high,93491


In [13]:
df[['question', 'value', 'difficulty']].head(10)

,question,value,difficulty
0,"'For the last 8 years of his life, Galileo was...",200.0,low
1,'No. 2: 1912 Olympian; football star at Carlis...,200.0,low
2,'The city of Yuma in this state has a record a...,200.0,low
3,"'In 1963, live on ""The Art Linkletter Show"", t...",200.0,low
4,"'Signer of the Dec. of Indep., framer of the C...",200.0,low
5,"'In the title of an Aesop fable, this insect s...",200.0,low
6,'Built in 312 B.C. to link Rome & the South of...,400.0,low
7,'No. 8: 30 steals for the Birmingham Barons; 2...,400.0,low
8,"'In the winter of 1971-72, a record 1,122 inch...",400.0,low
9,'This housewares store was named for the packa...,400.0,low


In [14]:
X = df['question']
y = df['difficulty']

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(len(X_train))
print(len(X_test))

170636
42660


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(170636, 81894)


In [18]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [19]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.5880450070323489


In [23]:
baseline = y_test.value_counts(normalize=True).max()

print("Baseline accuracy:", baseline)
print("Model accuracy:", accuracy)

Baseline accuracy: 0.5645100796999531
Model accuracy: 0.5880450070323489
